# BiGRU + Conv1D with GWO Hyperparameter Optimisation
## Unified Pipeline: Baseline → GWO Search → Optimised Final Training




## 1. Environment & Imports

In [10]:
# ============================================================
# 1.1  Environment Variables
# ============================================================
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"         # suppress TF C++ logs
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"        # deterministic ops
os.environ["CUDA_VISIBLE_DEVICES"]  = "0"      # adjust as needed

# ============================================================
# 1.2  Imports
# ============================================================
import warnings, random, time
from datetime import timedelta
from typing import Dict, List, Tuple

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, Conv1D, BatchNormalization, MaxPooling1D,
    GRU, Bidirectional, Dense, Dropout
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

from sklearn.metrics import (
    confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score
)

import matplotlib.pyplot as plt
import seaborn as sns

# mealpy >= 3.0  (pip install mealpy)
from mealpy.swarm_based.GWO import OriginalGWO
from mealpy.utils.space import IntegerVar, FloatVar

print("\u2713 All imports successful")
print(f"  TensorFlow : {tf.__version__}")
print(f"  NumPy      : {np.__version__}")

# ============================================================
# 1.3  Seed Utility
# ============================================================
def set_seeds(seed: int = RANDOM_STATE) -> None:
    """Fix all random seeds for reproducibility."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


# ============================================================
# 1.4  GPU Memory-Growth
# ============================================================
def setup_gpu() -> None:
    """Enable memory growth so TF does not grab all VRAM upfront."""
    gpus = tf.config.list_physical_devices("GPU")
    if not gpus:
        print("  \u26a0  No GPU detected \u2013 running on CPU.")
        return
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logical = tf.config.list_logical_devices("GPU")
        print(f"  \u2713  {len(gpus)} physical GPU(s) | {len(logical)} logical GPU(s)")
    except RuntimeError as exc:
        print(f"  \u26a0  GPU setup error: {exc}")


set_seeds()
setup_gpu()
print("\u2713 Reproducibility and GPU configuration complete.")

✓ All imports successful
  TensorFlow : 2.21.0
  NumPy      : 1.26.0
  ⚠  No GPU detected – running on CPU.
✓ Reproducibility and GPU configuration complete.


## 2. Global Configuration

In [19]:
# ============================================================
# 2.1  Paths
# ============================================================
DATA_PATHS: Dict[str, str] = {
    "train": "/home/gibannn/kuliah/sem3/paper/SMILES2VEC/code/SMILES/train_set_balanced.xlsx",
    "val"  : "/home/gibannn/kuliah/sem3/paper/SMILES2VEC/code/SMILES/val_set_balanced.xlsx",
    "test" : "/home/gibannn/kuliah/sem3/paper/SMILES2VEC/code/SMILES/test_set_balanced.xlsx",
}

MODEL_DIR   = "./models"
RESULTS_DIR = "./results"
PLOTS_DIR   = "./plots"
GWO_DIR     = "./gwo_results"

for d in (MODEL_DIR, RESULTS_DIR, PLOTS_DIR, GWO_DIR):
    os.makedirs(d, exist_ok=True)

# ============================================================
# 2.2  Training Constants
# ============================================================
MODEL_NAME   = "BiGRU_Conv"
RANDOM_STATE = 116
EPOCHS       = 100     # full run; EarlyStopping will cut short
PATIENCE     = 5

# Baseline reference parameters (fixed, for comparison)
BASELINE_PARAMS: Dict = {
    "gru_units"    : 64,
    "cnn_filters"  : 64,
    "learning_rate": 0.001,
    "batch_size"   : 116,
}

# ============================================================
# 2.3  GWO Configuration
# ============================================================
GWO_CONFIG: Dict = {
    "pop_size"   : 10,
    "epoch"      : 1,   # total evaluations ≈ pop_size × epoch
    "fit_epochs" : 30,   # reduced epochs per GWO fitness call
    "fit_patience": 3,   # early-stop patience during search
}

print(f"\u2713 Configuration ready  |  model={MODEL_NAME}  |  seed={RANDOM_STATE}")

✓ Configuration ready  |  model=BiGRU_Conv  |  seed=116


## 3. Data Loading

In [12]:
# ============================================================
# 4.1  Loader
# ============================================================
def load_dataset(path: str, split: str) -> Tuple[np.ndarray, np.ndarray]:
    """
    Load an Excel dataset.

    Expects columns: [feature_0, ..., feature_n, labels]

    Returns
    -------
    X : (n_samples, seq_len)   y : (n_samples,)
    """
    df = pd.read_excel(path)
    X  = df.drop(columns="labels").values
    y  = df["labels"].values
    print(f"  {split.upper():5s} -> X={X.shape}  y={y.shape}  "
          f"class counts={np.bincount(y.astype(int))}")
    return X, y


# ============================================================
# 4.2  Load All Splits Once (shared by GWO objective)
# ============================================================
print("Loading datasets \u2026")
X_train, y_train = load_dataset(DATA_PATHS["train"], "train")
X_val,   y_val   = load_dataset(DATA_PATHS["val"],   "val")
X_test,  y_test  = load_dataset(DATA_PATHS["test"],  "test")

VOCAB_SIZE = int(max(X_train.max(), X_val.max(), X_test.max()))
MAX_LEN    = int(X_train.shape[1])

print(f"\n\u2713 Vocab size : {VOCAB_SIZE}")
print(f"\u2713 Seq length : {MAX_LEN}")
print("\u2713 Data loaded once \u2013 will be reused for GWO.")

Loading datasets …
  TRAIN -> X=(8260, 271)  y=(8260,)  class counts=[4167 4093]
  VAL   -> X=(1806, 271)  y=(1806,)  class counts=[893 913]
  TEST  -> X=(1833, 271)  y=(1833,)  class counts=[893 940]

✓ Vocab size : 132
✓ Seq length : 271
✓ Data loaded once – will be reused for GWO.


## 5. Model Architecture

In [13]:
# ============================================================
# 5.1  Model Builder
# ============================================================
def build_model(
    gru_units    : int   = 64,
    cnn_filters  : int   = 64,
    dropout      : float = 0.3,
    learning_rate: float = 0.001,
    kernel_size  : int   = 5,
) -> Sequential:
    
    # coerce types (safe for direct calls and GWO callbacks)
    gru_units     = max(2,  int(round(float(gru_units))))
    cnn_filters   = max(1,  int(round(float(cnn_filters))))
    learning_rate = float(learning_rate)
    kernel_size   = int(kernel_size)

    set_seeds()   # deterministic weight initialisation

    model = Sequential([
        Embedding(
            input_dim=VOCAB_SIZE + 1,
            output_dim=128,
            input_length=MAX_LEN,
            mask_zero=True,           # propagate padding masks
        ),
        Conv1D(
            filters=cnn_filters,
            kernel_size=kernel_size,
            padding="same",
            activation="relu",
        ),
        BatchNormalization(),         # stabilises training
        MaxPooling1D(pool_size=2),
        Bidirectional(GRU(gru_units // 2, return_sequences=True, dropout=dropout)),
        Bidirectional(GRU(gru_units,      return_sequences=False, dropout=dropout)),
        Dropout(dropout),             # extra regularisation after BiGRU
        Dense(1, activation="sigmoid"),
    ], name=MODEL_NAME)

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss=tf.keras.losses.BinaryFocalCrossentropy(gamma=2),
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],
    )
    return model


print("\u2713 build_model() defined.")

# Quick architecture sanity-check
_tmp = build_model()
_tmp.summary()
del _tmp

✓ build_model() defined.


Model: "BiGRU_Conv"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## 6. Callbacks Factory

In [14]:
# ============================================================
# 6.1  Reusable Callback Builder
# ============================================================
def make_callbacks(
    checkpoint_path: str,
    monitor        : str  = "val_auc",
    patience       : int  = PATIENCE,
    verbose        : int  = 1,
) -> List:
    """
    Return [EarlyStopping, ModelCheckpoint] for a given monitor metric.

    The checkpoint saves the BEST model seen during training
    (save_best_only=True), so training can be safely interrupted.

    Parameters
    ----------
    checkpoint_path : str  – full .keras filepath
    monitor         : str  – metric to watch (default: val_auc)
    patience        : int  – early-stopping patience
    verbose         : int  – 0 = silent, 1 = print on improvement
    """
    return [
        EarlyStopping(
            monitor=monitor,
            mode="max",
            patience=patience,
            restore_best_weights=True,
            verbose=verbose,
        ),
        ModelCheckpoint(
            filepath=checkpoint_path,
            monitor=monitor,
            mode="max",
            save_best_only=True,
            verbose=verbose,
        ),
    ]


print("\u2713 make_callbacks() defined.")

✓ make_callbacks() defined.


## 7. Baseline Model Training

In [15]:
# ============================================================
# 7.1  Build & Train
# ============================================================
print("=" * 65)
print(" BASELINE MODEL TRAINING")
print("=" * 65)

tf.keras.backend.clear_session()
set_seeds()

baseline_model = build_model(
    gru_units    =BASELINE_PARAMS["gru_units"],
    cnn_filters  =BASELINE_PARAMS["cnn_filters"],
    learning_rate=BASELINE_PARAMS["learning_rate"],
)
baseline_model.summary()

baseline_ckpt = os.path.join(MODEL_DIR, f"{MODEL_NAME}_baseline.keras")

_t0 = time.time()
baseline_history = baseline_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs    =EPOCHS,
    batch_size=BASELINE_PARAMS["batch_size"],
    callbacks =make_callbacks(baseline_ckpt),
    verbose   =2,
)
baseline_time = time.time() - _t0

print(f"\n\u2713 Baseline training finished in {timedelta(seconds=int(baseline_time))}")
print(f"\u2713 Best checkpoint saved \u2192 {baseline_ckpt}")

 BASELINE MODEL TRAINING


Model: "BiGRU_Conv"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100

Epoch 1: val_auc improved from None to 0.95627, saving model to ./models/BiGRU_Conv_baseline.keras
72/72 - 36s - 494ms/step - accuracy: 0.8208 - auc: 0.8974 - loss: 0.1021 - val_accuracy: 0.8726 - val_auc: 0.9563 - val_loss: 0.1320
Epoch 2/100

Epoch 2: val_auc improved from 0.95627 to 0.96366, saving model to ./models/BiGRU_Conv_baseline.keras
72/72 - 25s - 343ms/step - accuracy: 0.9046 - auc: 0.9543 - loss: 0.0643 - val_accuracy: 0.8616 - val_auc: 0.9637 - val_loss: 0.1053
Epoch 3/100

Epoch 3: val_auc improved from 0.96366 to 0.96612, saving model to ./models/BiGRU_Conv_baseline.keras
72/72 - 23s - 317ms/step - accuracy: 0.9186 - auc: 0.9640 - loss: 0.0555 - val_accuracy: 0.8632 - val_auc: 0.9661 - val_loss: 0.0920
Epoch 4/100

Epoch 4: val_auc improved from 0.96612 to 0.96933, saving model to ./models/BiGRU_Conv_baseline.keras
72/72 - 22s - 311ms/step - accuracy: 0.9203 - auc: 0.9675 - loss: 0.0532 - val_accuracy: 0.9053 - val_auc: 0.9693 - val_loss: 0.0688
Epoch 5/100

In [16]:
# ============================================================
# 7.2  Quick Validation Metrics
# ============================================================
_prob = baseline_model.predict(X_val, verbose=0).flatten()
baseline_val_pred = (_prob > 0.5).astype(int)

baseline_val_f1  = f1_score(y_val, baseline_val_pred)
baseline_val_acc = accuracy_score(y_val, baseline_val_pred)

print("\n" + "=" * 65)
print(" BASELINE \u2013 Validation Results")
print("=" * 65)
print(f"  Accuracy  : {baseline_val_acc:.4f}")
print(f"  F1-Score  : {baseline_val_f1:.4f}")
print("=" * 65)


 BASELINE – Validation Results
  Accuracy  : 0.9369
  F1-Score  : 0.9359


## 8. Grey Wolf Optimizer (GWO) – Hyperparameter Search

In [6]:
# ============================================================
# 8.1  Search-Space Definition 
# Variable order: [gru_units, cnn_filters, dropout, learning_rate, batch_size]
# ============================================================
from niapy.algorithms.basic import GreyWolfOptimizer
from niapy.problems import Problem
from niapy.task import Task

class GWOProblem(Problem):
    def __init__(self, task: Task):
        super().__init__(task)
        self.bounds = [
            IntegerVar("gru_units", 16, 128),
            IntegerVar("cnn_filters", 16, 128),
            FloatVar("dropout", 0.1, 0.5),
            FloatVar("learning_rate", 1e-4, 1e-2),
            IntegerVar("batch_size", 32, 256),
        ]
    def evaluate(self, solution: List) -> float:
        gru_units, cnn_filters, dropout, learning_rate, batch_size = solution
        model = build_model(
            gru_units=gru_units,
            cnn_filters=cnn_filters,
            dropout=dropout,
            learning_rate=learning_rate,
        )
        history = model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=GWO_CONFIG["fit_epochs"],
            batch_size=batch_size,
            callbacks=make_callbacks(
                checkpoint_path=os.path.join(GWO_DIR, "temp.keras"),
                monitor="val_auc",
                patience=GWO_CONFIG["fit_patience"],
                verbose=0,
            ),
            verbose=0,
        )
        val_auc = max(history.history["val_auc"])
        return -val_auc  # GWO minimizes, so negate AUC    



def print_gwo_space(bounds: List) -> None:
    print("\u2713 GWO search space:")
    for v in bounds:
        print(f"   {v.name:20s}: [{v.lb}, {v.ub}]")


NameError: name 'List' is not defined

In [32]:
# ============================================================
# 8.2  Objective Function  (minimise -val_auc)
# ============================================================
_gwo_iter_count = 0
_gwo_best_auc   = 0.0
gwo_log: List[Dict] = []


def gwo_objective(solution: np.ndarray) -> float:
    """
    GWO objective: returns negative validation AUC (minimisation).

    Parameters
    ----------
    solution : [gru_units, cnn_filters, dropout, learning_rate, batch_size]

    Returns
    -------
    float  \u2013  -val_auc
    """
    global _gwo_iter_count, _gwo_best_auc
    _gwo_iter_count += 1

    gru_units     = max(2,  int(round(float(solution[0]))))
    cnn_filters   = max(1,  int(round(float(solution[1]))))
    dropout       = float(np.clip(solution[2], 0.0, 0.9))
    learning_rate = float(solution[3])
    batch_size    = max(16, int(round(float(solution[4]))))

    print(f"[Gwo iter {_gwo_iter_count:3d}] LR={learning_rate:.2e}  Batch={batch_size}  GRU={gru_units}  Drop={dropout:.3f}")

    try:
        tf.keras.backend.clear_session()
        set_seeds()

        model = build_model(
            gru_units    =gru_units,
            cnn_filters  =cnn_filters,
            dropout      =dropout,
            learning_rate=learning_rate,
        )

        history = model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs    =GWO_CONFIG["fit_epochs"],
            batch_size=batch_size,
            callbacks =[
                EarlyStopping(
                    monitor="val_auc", mode="max",
                    patience=GWO_CONFIG["fit_patience"],
                    restore_best_weights=True, verbose=0,
                )
            ],
            verbose=0,
        )

        val_auc = float(max(history.history["val_auc"]))
        val_loss = float(min(history.history["val_loss"]))
        val_f1  = f1_score(
            y_val,
            (model.predict(X_val, verbose=0) > 0.5).astype(int).flatten(),
            zero_division=0,
        )

        fitness = -(val_auc - 0.1 * val_loss)

        if val_auc > _gwo_best_auc:
            _gwo_best_auc = val_auc
            print(f"  NEW BEST  F1={val_f1:.4f}  Acc={val_auc:.4f}  ValLoss={val_loss:.4f}")
        else:
            print(f"  F1={val_f1:.4f}  Acc={val_auc:.4f}  ValLoss={val_loss:.4f}")

        gwo_log.append({
            "iter": _gwo_iter_count,
            "gru_units": gru_units, 
            "cnn_filters": cnn_filters,
            "learning_rate": learning_rate,
            "batch_size": batch_size,
            "val_auc": val_auc, "val_f1": val_f1, "val_loss": val_loss,
            "n_epochs": len(history.history["loss"]),
            "train_loss": float(history.history["loss"][-1]),
        })
        return fitness

    except Exception as exc:
        print(f"  \u2717 GWO iteration error: {exc}")
        return 0.0


print("\u2713 gwo_objective() defined.")

✓ gwo_objective() defined.


In [33]:
# ============================================================
# 8.3  Run GWO
# ============================================================
print("\n" + "=" * 65)
print(" STARTING GWO OPTIMISATION")
print(f"  pop_size={GWO_CONFIG['pop_size']}  "
      f"epochs={GWO_CONFIG['epoch']}  "
      f"total evals={GWO_CONFIG['pop_size'] * GWO_CONFIG['epoch']}")
print("=" * 65)

problem = {
    "obj_func": gwo_objective,
    "bounds"  : GWO_BOUNDS,
    "minmax"  : "min",
}

gwo = OriginalGWO(epoch=GWO_CONFIG["epoch"], pop_size=GWO_CONFIG["pop_size"])

_gwo_t0  = time.time()
g_best   = gwo.solve(problem)
gwo_time = time.time() - _gwo_t0

best_solution = g_best.solution
best_gwo_auc  = -g_best.target.fitness

best_entry = max(gwo_log, key=lambda x: x["val_auc"])
best_val_loss = best_entry["val_loss"]
best_entry = max(gwo_log, key=lambda x: x["val_auc"])
best_val_loss = best_entry["val_loss"]

print("\n" + "=" * 65)
print(" GWO COMPLETED")
print(f"  Time     : {timedelta(seconds=int(gwo_time))}")
print(f"  Best AUC : {best_gwo_auc:.4f}")
print(f"  Best loss: {best_val_loss:.4f}")
print(f"  Best loss: {best_val_loss:.4f}")
print("=" * 65)

2026/05/02 07:47:54 PM, INFO, mealpy.swarm_based.GWO.OriginalGWO: OriginalGWO(epoch=1, pop_size=10)



 STARTING GWO OPTIMISATION
  pop_size=10  epochs=1  total evals=10
[Gwo iter   1] LR=7.07e-03  Batch=101  GRU=86  Drop=0.240
  NEW BEST  F1=0.9251  Acc=0.9796  ValLoss=0.0433
[Gwo iter   2] LR=2.89e-03  Batch=50  GRU=43  Drop=0.500
  F1=0.9261  Acc=0.9723  ValLoss=0.0479
[Gwo iter   3] LR=6.69e-03  Batch=86  GRU=93  Drop=0.185
  NEW BEST  F1=0.9316  Acc=0.9796  ValLoss=0.0446
[Gwo iter   4] LR=9.13e-03  Batch=71  GRU=126  Drop=0.082
  NEW BEST  F1=0.9146  Acc=0.9798  ValLoss=0.0451
[Gwo iter   5] LR=8.80e-03  Batch=100  GRU=67  Drop=0.704
  F1=0.9176  Acc=0.9661  ValLoss=0.0649
[Gwo iter   6] LR=7.35e-03  Batch=44  GRU=67  Drop=0.693
  F1=0.9203  Acc=0.9658  ValLoss=0.0564
[Gwo iter   7] LR=5.49e-03  Batch=79  GRU=46  Drop=0.374
  F1=0.9266  Acc=0.9739  ValLoss=0.0466
[Gwo iter   8] LR=3.79e-04  Batch=108  GRU=75  Drop=0.702


KeyboardInterrupt: 

In [ ]:
# ============================================================
# 8.4  Extract & Persist Best Hyper-parameters
# ============================================================
best_params: Dict = {
    "gru_units"    : max(2,  int(round(float(best_solution[0])))),
    "cnn_filters"  : max(1,  int(round(float(best_solution[1])))),
    "learning_rate": float(best_solution[3]),
    "batch_size"   : max(16, int(round(float(best_solution[4])))),
}

print("\n" + "=" * 65)
print(" BEST HYPER-PARAMETERS (GWO)")
print("=" * 65)
for k, v in best_params.items():
    print(f"  {k:20s} : {v}")
print("=" * 65)

gwo_log_df = pd.DataFrame(gwo_log)
_log_path  = os.path.join(GWO_DIR, f"{MODEL_NAME}_gwo_log.csv")
_prm_path  = os.path.join(GWO_DIR, f"{MODEL_NAME}_best_params.csv")
gwo_log_df.to_csv(_log_path, index=False)
pd.DataFrame([best_params]).to_csv(_prm_path, index=False)

print(f"\n\u2713 Optimisation log \u2192 {_log_path}")
print(f"\u2713 Best params      \u2192 {_prm_path}")

## 9. Final Model Training (GWO-Optimised Hyper-parameters)

In [ ]:
# ============================================================
# 9.1  Retrain with Best Params – Full Epochs + ModelCheckpoint
# ============================================================
print("=" * 65)
print(" FINAL MODEL TRAINING  (GWO-Optimised)")
print("=" * 65)

tf.keras.backend.clear_session()
set_seeds()

optimised_model = build_model(
    gru_units    =best_params["gru_units"],
    cnn_filters  =best_params["cnn_filters"],
    learning_rate=best_params["learning_rate"],
)
optimised_model.summary()

opt_ckpt = os.path.join(MODEL_DIR, f"{MODEL_NAME}_optimised.keras")

_t0 = time.time()
optimised_history = optimised_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs    =EPOCHS,
    batch_size=best_params["batch_size"],
    callbacks =make_callbacks(opt_ckpt),
    verbose   =2,
)
optimised_time = time.time() - _t0

print(f"\n\u2713 Training finished in {timedelta(seconds=int(optimised_time))}")
print(f"\u2713 Best checkpoint saved \u2192 {opt_ckpt}")

## 10. Evaluation

In [ ]:
# ============================================================
# 10.1  Metric Helper
# ============================================================
def evaluate_model(
    model,
    X       : np.ndarray,
    y_true  : np.ndarray,
    split   : str   = "test",
    threshold: float = 0.5,
) -> Dict:
    """Predict and compute full classification metrics."""
    y_prob = model.predict(X, verbose=0).flatten()
    y_pred = (y_prob > threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "split"    : split,
        "Accuracy" : accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall"   : recall_score   (y_true, y_pred, zero_division=0),
        "F1"       : f1_score       (y_true, y_pred, zero_division=0),
        "TP": int(tp), "TN": int(tn), "FP": int(fp), "FN": int(fn),
    }


print("\u2713 evaluate_model() defined.")

In [ ]:
# ============================================================
# 10.2  Compute Metrics
# ============================================================
base_val  = evaluate_model(baseline_model,  X_val,  y_val,  "val")
base_test = evaluate_model(baseline_model,  X_test, y_test, "test")
opt_val   = evaluate_model(optimised_model, X_val,  y_val,  "val")
opt_test  = evaluate_model(optimised_model, X_test, y_test, "test")

for label, m_val, m_test in [
    ("BASELINE",      base_val, base_test),
    ("GWO-OPTIMISED", opt_val,  opt_test),
]:
    print("\n" + "=" * 65)
    print(f" {label}")
    print("=" * 65)
    for m in (m_val, m_test):
        s = m["split"].upper()
        print(f"  [{s:4s}] Acc={m['Accuracy']:.4f}  "
              f"Pre={m['Precision']:.4f}  "
              f"Rec={m['Recall']:.4f}  "
              f"F1={m['F1']:.4f}")
print("=" * 65)

In [ ]:
# ============================================================
# 10.3  Save Comparison Table
# ============================================================
rows = []
for label, params, m_test, t in [
    ("Baseline",      BASELINE_PARAMS, base_test, baseline_time),
    ("GWO-Optimised", best_params,     opt_test,  optimised_time),
]:
    row = {"Model": label}
    row.update({k: v for k, v in params.items()})
    row.update({k: v for k, v in m_test.items() if k != "split"})
    row["train_time"] = str(timedelta(seconds=int(t)))
    rows.append(row)

comparison_df = pd.DataFrame(rows)
_cmp_path = os.path.join(RESULTS_DIR, f"{MODEL_NAME}_comparison.csv")
comparison_df.to_csv(_cmp_path, index=False)
print(f"\u2713 Comparison table \u2192 {_cmp_path}")
print(comparison_df.to_string(index=False))

# % improvements
for metric in ("F1", "Accuracy"):
    b, o = comparison_df.loc[0, metric], comparison_df.loc[1, metric]
    delta = (o - b) / b * 100 if b > 0 else float("nan")
    print(f"  Improvement {metric:10s}: {delta:+.2f}%")

# Per-model metrics CSV  (Notebook B compatibility)
for lbl, m in [("baseline", base_test), ("optimised", opt_test)]:
    p = os.path.join(RESULTS_DIR, f"{MODEL_NAME}_{lbl}_metrics.csv")
    pd.DataFrame([m]).to_csv(p, index=False)
    print(f"  \u2713 {p}")

## 11. Visualisations

In [ ]:
# ============================================================
# 11.1  Training History Comparison
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, metric, ylabel in [
    (axes[0], "val_accuracy", "Accuracy"),
    (axes[1], "val_auc",      "AUC"),
]:
    ax.plot(baseline_history.history[metric],  label="Baseline",      lw=2, ls="--")
    ax.plot(optimised_history.history[metric], label="GWO-Optimised", lw=2)
    ax.set(title=f"Validation {ylabel}", xlabel="Epoch", ylabel=ylabel)
    ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
_p = os.path.join(PLOTS_DIR, f"{MODEL_NAME}_training_history.png")
plt.savefig(_p, dpi=300, bbox_inches="tight"); plt.show()
print(f"\u2713 Saved \u2192 {_p}")

In [ ]:
# ============================================================
# 11.2  Confusion Matrix Comparison
# ============================================================
cm_base = confusion_matrix(y_test, (baseline_model.predict(X_test, verbose=0) > 0.5).astype(int))
cm_opt  = confusion_matrix(y_test, (optimised_model.predict(X_test, verbose=0) > 0.5).astype(int))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, cm, title, cmap in [
    (axes[0], cm_base, "Baseline",      "Blues"),
    (axes[1], cm_opt,  "GWO-Optimised", "Greens"),
]:
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap, ax=ax,
                xticklabels=["Neg", "Pos"], yticklabels=["Neg", "Pos"])
    ax.set(title=f"{title} \u2013 Confusion Matrix",
           xlabel="Predicted", ylabel="Actual")

plt.tight_layout()
_p = os.path.join(PLOTS_DIR, f"{MODEL_NAME}_confusion_matrices.png")
plt.savefig(_p, dpi=300, bbox_inches="tight"); plt.show()
print(f"\u2713 Saved \u2192 {_p}")

In [ ]:
# ============================================================
# 11.3  Metrics Bar Chart
# ============================================================
_mnames = ["Accuracy", "Precision", "Recall", "F1"]
x, w = np.arange(len(_mnames)), 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars_b = ax.bar(x - w/2, [base_test[m] for m in _mnames], w,
                label="Baseline",      color="#3498db", alpha=0.85)
bars_o = ax.bar(x + w/2, [opt_test[m]  for m in _mnames], w,
                label="GWO-Optimised", color="#2ecc71", alpha=0.85)

for bars in (bars_b, bars_o):
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h,
                f"{h:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

ax.set(xticks=x, xticklabels=_mnames, ylabel="Score",
       title="Test-Set Performance Comparison", ylim=[0, 1.05])
ax.legend(); ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
_p = os.path.join(PLOTS_DIR, f"{MODEL_NAME}_metrics_bar.png")
plt.savefig(_p, dpi=300, bbox_inches="tight"); plt.show()
print(f"\u2713 Saved \u2192 {_p}")

In [ ]:
# ============================================================
# 11.4  GWO Optimisation Progress  (4-panel)
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# AUC per iteration
axes[0, 0].plot(gwo_log_df["iter"], gwo_log_df["val_auc"], marker="o", lw=2, ms=5)
axes[0, 0].axhline(y=max(baseline_history.history["val_auc"]),
                   color="r", ls="--", lw=2, label="Baseline best AUC")
axes[0, 0].set(xlabel="GWO Iteration", ylabel="Val AUC", title="Optimisation Progress")
axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)

# GRU units vs AUC
sc1 = axes[0, 1].scatter(gwo_log_df["gru_units"], gwo_log_df["val_auc"],
                          c=gwo_log_df["learning_rate"], cmap="viridis", s=80, alpha=0.7)
axes[0, 1].set(xlabel="GRU Units", ylabel="Val AUC", title="GRU Units vs AUC")
plt.colorbar(sc1, ax=axes[0, 1], label="Learning Rate")
axes[0, 1].grid(True, alpha=0.3)

# CNN filters vs AUC
sc2 = axes[1, 0].scatter(gwo_log_df["cnn_filters"], gwo_log_df["val_auc"],
                          c=gwo_log_df["dropout"], cmap="plasma", s=80, alpha=0.7)
axes[1, 0].set(xlabel="CNN Filters", ylabel="Val AUC", title="CNN Filters vs AUC")
plt.colorbar(sc2, ax=axes[1, 0], label="Dropout Rate")
axes[1, 0].grid(True, alpha=0.3)

# Batch size vs AUC
axes[1, 1].scatter(gwo_log_df["batch_size"], gwo_log_df["val_auc"],
                   color="#9b59b6", s=80, alpha=0.7)
axes[1, 1].set(xlabel="Batch Size", ylabel="Val AUC", title="Batch Size vs AUC")
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
_p = os.path.join(PLOTS_DIR, f"{MODEL_NAME}_gwo_progress.png")
plt.savefig(_p, dpi=300, bbox_inches="tight"); plt.show()
print(f"\u2713 Saved \u2192 {_p}")